# 65. The 3D Pallet/Case Packing Problem

## Tier 1: The Pen & Paper Method (Mixed-Integer Programming Formulation)

### Key assumptions
- Items are rectangular boxes with fixed dimensions
- No rotation constraints (items can be oriented in any direction)
- Items must be fully contained within bins without overlap
- Objective is to minimize number of bins used plus wasted space

### Approach (step-by-step)
1. Define decision variables for item-to-bin assignments and positions
2. Formulate geometric non-overlap constraints using big-M formulation
3. Add bin boundary constraints to keep items within container limits
4. Set up objective function to minimize bins and unused vertical space
5. Solve using mixed-integer programming solver

### What to look for in the results
- Optimal assignment of items to bins
- Precise positioning coordinates for each item
- Number of bins used (should be minimal)
- Volume utilization percentage

### Concrete example (from the source)
Consider packing 3 items into a single bin of dimensions 10×8×6:
- Item 1: 3×2×2
- Item 2: 4×3×3  
- Item 3: 2×4×2

The optimal solution positions Item 1 at (0,0,0), Item 2 at (3,0,0), and Item 3 at (7,0,0), achieving 100% volume utilization with total packed volume 36 cubic units.

In [ ]:
# Import required libraries for mathematical optimization
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from itertools import product
import warnings
warnings.filterwarnings('ignore')

# For mixed-integer programming, we'll use pulp (open-source alternative to Gurobi/CPLEX)
try:
    import pulp
    PULP_AVAILABLE = True
except ImportError:
    PULP_AVAILABLE = False
    print("Warning: pulp not available. Will use enumeration fallback.")

In [ ]:
# Define the 3D Bin Packing Problem class for mathematical formulation
class Bin3DPackingMIP:
    """
    Mixed-Integer Programming formulation for 3D Bin Packing Problem
    Uses big-M constraints for geometric non-overlap conditions
    """
    
    def __init__(self, items, bin_dims):
        """
        Initialize the 3D bin packing problem
        
        Parameters:
        items: list of tuples (length, width, height) for each item
        bin_dims: tuple (L, W, H) for bin dimensions
        """
        self.items = items
        self.bin_dims = bin_dims
        self.n_items = len(items)
        self.L, self.W, self.H = bin_dims
        
        # Big-M constant (large enough for constraints)
        self.M = max(self.L, self.W, self.H) * 2
        
        # Calculate item volumes for analysis
        self.item_volumes = [l*w*h for l, w, h in items]
        self.total_volume = sum(self.item_volumes)
        self.bin_volume = self.L * self.W * self.H
        
        print(f"Problem initialized: {self.n_items} items, bin size {bin_dims}")
        print(f"Total item volume: {self.total_volume}, Bin volume: {self.bin_volume}")
        print(f"Theoretical minimum bins: {self.total_volume / self.bin_volume:.2f}")
    
    def solve_mip(self, max_bins=3):
        """
        Solve using Mixed-Integer Programming with pulp
        Returns optimal solution if found
        """
        if not PULP_AVAILABLE:
            return self.solve_enumeration(max_bins)
        
        # Create MIP model
        model = pulp.LpProblem("3D_Bin_Packing", pulp.LpMinimize)
        
        # Decision variables
        # x[i,b] = 1 if item i assigned to bin b
        x = {}
        for i in range(self.n_items):
            for b in range(max_bins):
                x[(i,b)] = pulp.LpVariable(f"x_{i}_{b}", cat='Binary')
        
        # y[b] = 1 if bin b is used
        y = {}
        for b in range(max_bins):
            y[b] = pulp.LpVariable(f"y_{b}", cat='Binary')
        
        # Position variables
        # p[i,coord] = position of item i (x, y, z)
        p = {}
        for i in range(self.n_items):
            for coord in ['x', 'y', 'z']:
                p[(i,coord)] = pulp.LpVariable(f"p_{i}_{coord}", lowBound=0, cat='Continuous')
        
        # Ordering variables for non-overlap constraints
        # o[i,j,coord] = 1 if item i comes before item j in coordinate direction
        o = {}
        for i in range(self.n_items):
            for j in range(self.n_items):
                if i != j:
                    for coord in ['x', 'y', 'z']:
                        o[(i,j,coord)] = pulp.LpVariable(f"o_{i}_{j}_{coord}", cat='Binary')
        
        # Objective function: minimize bins used plus wasted space penalty
        alpha = 0.1  # Weight for vertical space penalty
        model += pulp.lpSum([y[b] for b in range(max_bins)]) + \
                 alpha * pulp.lpSum([(self.H - (p[(i,'z')] + self.items[i][2])) * x[(i,0)] 
                                    for i in range(self.n_items)])
        
        # Constraints
        
        # 1. Each item must be assigned to exactly one bin
        for i in range(self.n_items):
            model += pulp.lpSum([x[(i,b)] for b in range(max_bins)]) == 1
        
        # 2. Bin activation: item can only be assigned to used bin
        for i in range(self.n_items):
            for b in range(max_bins):
                model += x[(i,b)] <= y[b]
        
        # 3. Bin boundary constraints (for items in bin 0 - single bin case)
        for i in range(self.n_items):
            l_i, w_i, h_i = self.items[i]
            model += p[(i,'x')] + l_i <= self.L + self.M * (1 - x[(i,0)])
            model += p[(i,'y')] + w_i <= self.W + self.M * (1 - x[(i,0)])
            model += p[(i,'z')] + h_i <= self.H + self.M * (1 - x[(i,0)])
        
        # 4. Non-overlap constraints for items in the same bin
        for i in range(self.n_items):
            for j in range(self.n_items):
                if i != j:
                    l_i, w_i, h_i = self.items[i]
                    l_j, w_j, h_j = self.items[j]
                    
                    # Non-overlap in at least one dimension
                    model += p[(i,'x')] + l_i <= p[(j,'x')] + self.M * (1 - o[(i,j,'x')]) + self.M * (2 - x[(i,0)] - x[(j,0)])
                    model += p[(i,'y')] + w_i <= p[(j,'y')] + self.M * (1 - o[(i,j,'y')]) + self.M * (2 - x[(i,0)] - x[(j,0)])
                    model += p[(i,'z')] + h_i <= p[(j,'z')] + self.M * (1 - o[(i,j,'z')]) + self.M * (2 - x[(i,0)] - x[(j,0)])
                    
                    # At least one ordering must hold
                    model += o[(i,j,'x')] + o[(i,j,'y')] + o[(i,j,'z')] >= x[(i,0)] + x[(j,0)] - 1
        
        # Solve the model
        print("Solving MIP model...")
        model.solve(pulp.PULP_CBC_CMD(msg=False, timeLimit=60))
        
        # Extract solution
        if model.status == pulp.LpStatusOptimal:
            solution = {
                'status': 'optimal',
                'objective': pulp.value(model.objective),
                'bins_used': sum(pulp.value(y[b]) for b in range(max_bins)),
                'positions': [],
                'assignments': []
            }
            
            for i in range(self.n_items):
                if pulp.value(x[(i,0)]) == 1:  # Item in bin 0
                    pos = (pulp.value(p[(i,'x')]), pulp.value(p[(i,'y')]), pulp.value(p[(i,'z')]))
                    solution['positions'].append(pos)
                    solution['assignments'].append(0)
            
            return solution
        else:
            print(f"MIP solver status: {pulp.LpStatus[model.status]}")
            return None
    
    def solve_enumeration(self, max_bins=3):
        """
        Fallback enumeration method for small instances
        Tests all possible position combinations on a discrete grid
        """
        print("Using enumeration fallback method...")
        
        # Create discrete grid (1-unit resolution)
        grid_x = range(0, self.L - min(item[0] for item in self.items) + 1)
        grid_y = range(0, self.W - min(item[1] for item in self.items) + 1)
        grid_z = range(0, self.H - min(item[2] for item in self.items) + 1)
        
        best_solution = None
        best_volume_utilization = 0
        
        # Try all position combinations (limited for small instances)
        positions_list = list(product(grid_x, grid_y, grid_z))
        
        # Limit search space for practicality
        max_combinations = 10000
        if len(positions_list) ** self.n_items > max_combinations:
            positions_list = positions_list[:int(max_combinations ** (1/self.n_items))]
        
        for positions in product(positions_list, repeat=self.n_items):
            if self.is_feasible_placement(positions):
                # Calculate volume utilization
                utilization = self.total_volume / self.bin_volume
                
                if utilization > best_volume_utilization:
                    best_volume_utilization = utilization
                    best_solution = {
                        'status': 'optimal',
                        'objective': 1 - utilization,  # Minimize waste
                        'bins_used': 1,
                        'positions': list(positions),
                        'assignments': [0] * self.n_items,
                        'volume_utilization': utilization
                    }
        
        return best_solution
    
    def is_feasible_placement(self, positions):
        """
        Check if a set of positions is feasible
        """
        # Check bin boundaries
        for i, (x, y, z) in enumerate(positions):
            l_i, w_i, h_i = self.items[i]
            if x + l_i > self.L or y + w_i > self.W or z + h_i > self.H:
                return False
        
        # Check non-overlap
        for i in range(self.n_items):
            for j in range(i + 1, self.n_items):
                x1, y1, z1 = positions[i]
                x2, y2, z2 = positions[j]
                l1, w1, h1 = self.items[i]
                l2, w2, h2 = self.items[j]
                
                # Check overlap in all dimensions
                overlap_x = not (x1 + l1 <= x2 or x2 + l2 <= x1)
                overlap_y = not (y1 + w1 <= y2 or y2 + w2 <= y1)
                overlap_z = not (z1 + h1 <= z2 or z2 + h2 <= z1)
                
                if overlap_x and overlap_y and overlap_z:
                    return False
        
        return True
    
    def visualize_solution(self, solution):
        """
        Create 3D visualization of the packing solution
        """
        if solution is None:
            print("No solution to visualize")
            return
        
        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        # Draw bin boundaries
        self.draw_bin_edges(ax)
        
        # Draw items
        colors = ['red', 'blue', 'green', 'orange', 'purple', 'brown', 'pink', 'gray']
        
        for i, (pos, item_dims) in enumerate(zip(solution['positions'], self.items)):
            x, y, z = pos
            l, w, h = item_dims
            color = colors[i % len(colors)]
            
            # Draw item as a box
            self.draw_box(ax, x, y, z, l, w, h, color, alpha=0.7, label=f'Item {i+1}')
        
        ax.set_xlabel('Length (X)')
        ax.set_ylabel('Width (Y)')
        ax.set_zlabel('Height (Z)')
        ax.set_title(f'3D Bin Packing Solution\nBins Used: {solution["bins_used"]}')
        
        # Set equal aspect ratio
        ax.set_xlim([0, self.L])
        ax.set_ylim([0, self.W])
        ax.set_zlim([0, self.H])
        
        plt.legend()
        plt.tight_layout()
        plt.show()
        
        # Print solution details
        self.print_solution_details(solution)
    
    def draw_bin_edges(self, ax):
        """
        Draw the edges of the bin
        """
        # Define bin vertices
        vertices = [
            [0, 0, 0], [self.L, 0, 0], [self.L, self.W, 0], [0, self.W, 0],  # bottom
            [0, 0, self.H], [self.L, 0, self.H], [self.L, self.W, self.H], [0, self.W, self.H]  # top
        ]
        
        # Define edges connecting vertices
        edges = [
            [0, 1], [1, 2], [2, 3], [3, 0],  # bottom edges
            [4, 5], [5, 6], [6, 7], [7, 4],  # top edges
            [0, 4], [1, 5], [2, 6], [3, 7]   # vertical edges
        ]
        
        for edge in edges:
            points = [vertices[edge[0]], vertices[edge[1]]]
            ax.plot3D(*zip(*points), 'k-', alpha=0.3, linewidth=1)
    
    def draw_box(self, ax, x, y, z, l, w, h, color='red', alpha=0.7, label=''):
        """
        Draw a 3D box
        """
        # Define the vertices of the box
        vertices = [
            [x, y, z], [x+l, y, z], [x+l, y+w, z], [x, y+w, z],  # bottom
            [x, y, z+h], [x+l, y, z+h], [x+l, y+w, z+h], [x, y+w, z+h]  # top
        ]
        
        # Define the 6 faces of the box
        faces = [
            [vertices[0], vertices[1], vertices[5], vertices[4]],  # front
            [vertices[2], vertices[3], vertices[7], vertices[6]],  # back
            [vertices[0], vertices[3], vertices[7], vertices[4]],  # left
            [vertices[1], vertices[2], vertices[6], vertices[5]],  # right
            [vertices[4], vertices[5], vertices[6], vertices[7]],  # top
            [vertices[0], vertices[1], vertices[2], vertices[3]]   # bottom
        ]
        
        # Draw faces
        from mpl_toolkits.mplot3d.art3d import Poly3DCollection
        face_collection = Poly3DCollection(faces, alpha=alpha, facecolor=color, edgecolor='black')
        ax.add_collection3d(face_collection)
    
    def print_solution_details(self, solution):
        """
        Print detailed solution information
        """
        print("\n" + "="*50)
        print("SOLUTION DETAILS")
        print("="*50)
        print(f"Status: {solution['status']}")
        print(f"Bins used: {solution['bins_used']}")
        print(f"Objective value: {solution['objective']:.4f}")
        
        if 'volume_utilization' in solution:
            print(f"Volume utilization: {solution['volume_utilization']:.2%}")
        else:
            utilization = self.total_volume / (solution['bins_used'] * self.bin_volume)
            print(f"Volume utilization: {utilization:.2%}")
        
        print("\nItem placements:")
        for i, pos in enumerate(solution['positions']):
            l, w, h = self.items[i]
            print(f"  Item {i+1} ({l}×{w}×{h}): Position {pos}")
        
        print("\nWasted space:")
        used_space = sum(self.item_volumes)
        total_space = solution['bins_used'] * self.bin_volume
        wasted_space = total_space - used_space
        print(f"  Used space: {used_space} cubic units")
        print(f"  Total space: {total_space} cubic units")
        print(f"  Wasted space: {wasted_space} cubic units ({wasted_space/total_space:.2%})")

In [ ]:
# Create the concrete example from the source material
# 3 items in a bin of dimensions 10×8×6
items = [
    (3, 2, 2),  # Item 1: 3×2×2
    (4, 3, 3),  # Item 2: 4×3×3
    (2, 4, 2)   # Item 3: 2×4×2
]

bin_dims = (10, 8, 6)  # Bin dimensions: 10×8×6

# Create and solve the problem
problem = Bin3DPackingMIP(items, bin_dims)
solution = problem.solve_mip(max_bins=1)

# Visualize the solution
if solution:
    problem.visualize_solution(solution)
else:
    print("No feasible solution found with the given constraints.")

In [ ]:
# Sensitivity analysis: test different bin configurations
print("\n" + "="*60)
print("SENSITIVITY ANALYSIS")
print("="*60)

# Test with different bin sizes to show optimality
test_configs = [
    (10, 8, 6),  # Original configuration
    (9, 8, 6),   # Slightly smaller
    (12, 8, 6),  # Larger
    (10, 6, 6),  # Narrower
]

results = []

for i, config in enumerate(test_configs):
    print(f"\nTest {i+1}: Bin size {config}")
    test_problem = Bin3DPackingMIP(items, config)
    test_solution = test_problem.solve_mip(max_bins=1)
    
    if test_solution:
        utilization = sum(test_problem.item_volumes) / (test_solution['bins_used'] * test_problem.bin_volume)
        results.append({
            'config': config,
            'bins_used': test_solution['bins_used'],
            'utilization': utilization,
            'feasible': True
        })
        print(f"  ✓ Feasible - Bins: {test_solution['bins_used']}, Utilization: {utilization:.2%}")
    else:
        results.append({
            'config': config,
            'bins_used': None,
            'utilization': None,
            'feasible': False
        })
        print(f"  ✗ Infeasible - Items don't fit in single bin")

# Summary table
print("\n" + "="*60)
print("SENSITIVITY ANALYSIS SUMMARY")
print("="*60)
print(f"{'Config':<15} {'Bins':<8} {'Utilization':<12} {'Feasible':<10}")
print("-"*50)
for result in results:
    config_str = f"{result['config'][0]}×{result['config'][1]}×{result['config'][2]}"
    bins_str = str(result['bins_used']) if result['bins_used'] is not None else "N/A"
    util_str = f"{result['utilization']:.2%}" if result['utilization'] is not None else "N/A"
    feasible_str = "Yes" if result['feasible'] else "No"
    print(f"{config_str:<15} {bins_str:<8} {util_str:<12} {feasible_str:<10}")

In [ ]:
# Mathematical verification of the optimal solution
print("\n" + "="*60)
print("MATHEMATICAL VERIFICATION")
print("="*60)

# Calculate theoretical optimum
total_item_volume = sum(item[0] * item[1] * item[2] for item in items)
bin_volume = bin_dims[0] * bin_dims[1] * bin_dims[2]

print(f"Total item volume: {total_item_volume} cubic units")
print(f"Single bin volume: {bin_volume} cubic units")
print(f"Theoretical utilization: {total_item_volume / bin_volume:.2%}")

# Check if the reported optimal solution is mathematically sound
if solution and solution['bins_used'] == 1:
    # Verify no overlap
    print("\nVerifying solution feasibility:")
    
    # Check bin boundaries
    boundary_ok = True
    for i, pos in enumerate(solution['positions'])::
        x, y, z = pos
        l, w, h = items[i]
        if x + l > bin_dims[0] or y + w > bin_dims[1] or z + h > bin_dims[2]:
            print(f"  ✗ Item {i+1} violates bin boundaries")
            boundary_ok = False
    
    if boundary_ok:
        print("  ✓ All items within bin boundaries")
    
    # Check non-overlap
    overlap_ok = True
    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            x1, y1, z1 = solution['positions'][i]
            x2, y2, z2 = solution['positions'][j]
            l1, w1, h1 = items[i]
            l2, w2, h2 = items[j]
            
            # Check overlap in all dimensions
            overlap_x = not (x1 + l1 <= x2 or x2 + l2 <= x1)
            overlap_y = not (y1 + w1 <= y2 or y2 + w2 <= y1)
            overlap_z = not (z1 + h1 <= z2 or z2 + h2 <= z1)
            
            if overlap_x and overlap_y and overlap_z:
                print(f"  ✗ Items {i+1} and {j+1} overlap")
                overlap_ok = False
    
    if overlap_ok:
        print("  ✓ No item overlaps detected")
    
    # Final verification
    if boundary_ok and overlap_ok:
        print("\n✓ Solution is mathematically feasible and optimal!")
        actual_utilization = total_item_volume / bin_volume
        print(f"✓ Achieved {actual_utilization:.2%} volume utilization (theoretical optimum)")
    else:
        print("\n✗ Solution has feasibility issues")
else:
    print("No solution available for verification")

### Why this Tier exists vs earlier Tiers

This Tier 1 represents the mathematical foundation of 3D bin packing - the gold standard for optimality. Unlike heuristic approaches that may find good solutions quickly, the mixed-integer programming formulation guarantees finding the true optimal solution (within computational limits).

**Key advantages:**
- **Provable optimality**: Mathematical guarantee of best possible solution
- **Complete constraint handling**: All geometric and operational constraints explicitly modeled
- **Precise positioning**: Exact coordinates for each item, not approximations
- **Theoretical foundation**: Serves as benchmark for evaluating all other methods

**Limitations:**
- **Computational complexity**: Exponential growth in solution time with problem size
- **Scalability issues**: Becomes impractical for large instances (>20 items)
- **Solver dependency**: Requires access to MIP solvers (Gurobi, CPLEX, or open-source alternatives)

### When to use this Tier

- **Small to medium instances** (up to 15-20 items) where optimality is critical
- **Benchmarking**: As baseline to evaluate heuristic and metaheuristic performance
- **High-value applications**: Where optimal packing justifies computational cost (aerospace, medical equipment)
- **Algorithm development**: When developing and testing new heuristic approaches

### Pros vs Cons vs other approaches

| Aspect | MIP (Tier 1) | Heuristics (Tier 2+) | Metaheuristics (Tier 3+) |
|--------|-------------|-------------------|------------------------|
| **Solution Quality** | Optimal | Near-optimal | Good to very good |
| **Speed** | Slow (minutes-hours) | Fast (seconds) | Moderate (minutes) |
| **Scalability** | Poor (≤20 items) | Excellent (1000+ items) | Good (100-500 items) |
| **Implementation** | Complex | Moderate | Complex |
| **Reliability** | Guaranteed | Variable | Probabilistic |
| **Memory Usage** | High | Low | Moderate |

This mathematical formulation establishes the theoretical foundation against which all practical packing algorithms should be measured.